# S2 · Text representation and recurring-language audit

The primary representation is masked `use` text. Description is retained as a sensitivity check. Train-only hashed TF–IDF avoids vocabulary leakage; recurring exact 5-grams are removed in the residual representation without attributing them to a specific institution.

In [1]:
from pathlib import Path
import json, time
import pandas as pd
import numpy as np
import psutil
from IPython.display import display, Image

ROOT = Path.cwd().parent
OUTPUTS = ROOT / "outputs"
AUDIT = ROOT / "audit"
LOGS = ROOT / "logs"
LOGS.mkdir(exist_ok=True)
RUN_LOG = LOGS / "run_log.txt"

def checkpoint(label, *frames, started=None):
    elapsed = time.time() - started if started is not None else 0.0
    shapes = [getattr(x, "shape", None) for x in frames]
    nulls = []
    for frame in frames:
        if hasattr(frame, "isna"):
            nulls.append(round(float(frame.isna().mean(numeric_only=False).mean()), 6))
    rss = psutil.Process().memory_info().rss / 1024**3
    line = f"{label} | shapes={shapes} | mean_null_rates={nulls} | rss_gib={rss:.3f} | elapsed_s={elapsed:.3f}"
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as stream:
        stream.write(line + "\n")

t0 = time.time()
print(f"Audit root: {ROOT}")
checkpoint("setup", started=t0)

Audit root: <PROJECT_ROOT>/5_最终交付包/_rebuild/MA-Hackathon-Final-2026-09-04
setup | shapes=[] | mean_null_rates=[] | rss_gib=0.112 | elapsed_s=0.000


In [2]:
t0 = time.time()
diagnostics = pd.read_csv(OUTPUTS / "text_selection_diagnostics.csv")
shares = pd.read_csv(OUTPUTS / "boilerplate_share_distribution.csv")
phrases = pd.read_csv(OUTPUTS / "boilerplate_like_phrases.csv")
manifest = json.loads((AUDIT / "stage2_text_manifest.json").read_text())
display(diagnostics)
display(shares)
display(phrases.head(20))
public_manifest = {key: value for key, value in manifest.items() if key != "processed_private_path"}
public_manifest["processed_private_checkpoint"] = "omitted from external delivery"
print(json.dumps(public_manifest, indent=2))
checkpoint("S2 text outputs", diagnostics, shares, phrases, started=t0)

,field,diagnostic_sample_rows,random_same_sector_pairs,pair_zero_overlap_pct,pair_cosine_mean,pair_cosine_variance,sample_window_h_n,sample_window_h_variance,sample_window_h_median,sample_pool_size_median,representation_note,nonempty_pct,tokens_before_median,tokens_after_median,tokens_after_p90
0,use,66083,10000,6.44,0.259326,0.041326,62183,0.025582,0.246767,69.0,deterministic 5% training sample; hashed unigr...,96.9901,10.0,10.0,18.0
1,description,66083,10000,5.85,0.408937,0.028351,62183,0.014902,0.428507,69.0,deterministic 5% training sample; hashed unigr...,96.9901,89.0,89.0,161.0


,loans,nonzero_share_pct,p10,p50,p90,p99
0,1453846,38.850951,0.0,0.0,0.8,1.0


,masked_fivegram,training_document_frequency,training_document_pct,countries_present,interpretation
0,and other supplies to raise,52714,4.003542,3,cross-country recurring boilerplate-like phras...
1,to buy items to sell,39666,3.012568,13,cross-country recurring boilerplate-like phras...
2,etc to sell in her,39254,2.981277,12,cross-country recurring boilerplate-like phras...
3,canned goods personal care products,39122,2.971252,3,cross-country recurring boilerplate-like phras...
4,to sell in her store,32743,2.486777,33,cross-country recurring boilerplate-like phras...
5,buy items to sell like,31683,2.406272,6,cross-country recurring boilerplate-like phras...
6,to provide safe drinking water,25797,1.959240,3,cross-country recurring boilerplate-like phras...
7,to sell like canned goods,25670,1.949595,3,cross-country recurring boilerplate-like phras...
8,to buy water filter to,25538,1.939569,3,cross-country recurring boilerplate-like phras...
9,sell in her general store,24416,1.854355,13,cross-country recurring boilerplate-like phras...


{
  "processed_private_bytes": 48395741,
  "raw_idf_nondefault_features": 115885,
  "residual_idf_nondefault_features": 115732,
  "rss_gib_at_end": 0.54,
  "elapsed_seconds": 80.786,
  "processed_private_checkpoint": "omitted from external delivery"
}
S2 text outputs | shapes=[(2, 15), (1, 6), (409, 5)] | mean_null_rates=[0.0, 0.0, 0.0] | rss_gib=0.114 | elapsed_s=0.012


In [3]:
t0 = time.time()
description = pd.read_csv(OUTPUTS / "description_robustness.csv")
desc_identity = pd.read_csv(OUTPUTS / "description_pool_identity_check.csv")
desc_manifest = json.loads((AUDIT / "description_robustness_manifest.json").read_text())
assert desc_manifest["identity_passed"]
assert desc_manifest["identity_max_abs_error"] <= 1e-9
display(description)
print(f"Description identity maximum absolute error: {desc_manifest['identity_max_abs_error']:.3e}")
checkpoint("S2 description sensitivity", description, desc_identity, started=t0)

,representation,n,term,estimate,std_error,p_value,ci_low,ci_high
0,masked description hashed TF-IDF,1234124,C,0.445054,0.065709,2.213325e-08,0.312708,0.577399
1,masked description hashed TF-IDF,1234124,H,0.515361,0.036117,0.000000e+00,0.442617,0.588104
2,masked description hashed TF-IDF,1234124,CH,0.129794,0.020988,1.662754e-07,0.087522,0.172066
3,masked description hashed TF-IDF,1234124,V,0.136727,0.104012,1.953284e-01,-0.072764,0.346219
4,masked description hashed TF-IDF,1234124,G,-0.225185,0.071932,3.062174e-03,-0.370063,-0.080306
5,masked description hashed TF-IDF,1234124,VG,-0.013572,0.015165,3.755626e-01,-0.044115,0.016971


Description identity maximum absolute error: 1.721e-15
S2 description sensitivity | shapes=[(6, 8), (100, 5)] | mean_null_rates=[0.0, 0.0] | rss_gib=0.114 | elapsed_s=0.004
